# 00 — Dataset check: per-ETF date coverage

Loads the raw price panel (`data/raw/prices.parquet`, a wide `date × ticker` frame) and
reports the **start date**, **end date**, and coverage for every ETF.

- *start* = first date with a non-NaN price
- *end*   = last date with a non-NaN price
- *n_obs* = number of trading days actually observed
- *coverage* = n_obs / trading days in the panel over [start, end] (1.0 = no internal gaps)

In [ ]:
import pathlib
import pandas as pd

import common

ROOT = common.resolve_root()
prices = pd.read_parquet(ROOT / 'data' / 'raw' / 'prices.parquet')
prices = prices.sort_index()

panel_index = prices.index
print(f'panel: {prices.shape[1]} ETFs x {prices.shape[0]} trading days')
print(f'panel span: {panel_index.min().date()} -> {panel_index.max().date()}')

In [ ]:
# Per-ETF first/last valid date + coverage.
first = prices.apply(lambda s: s.first_valid_index())
last = prices.apply(lambda s: s.last_valid_index())
n_obs = prices.notna().sum()


def _span_days(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0
    return panel_index.slice_indexer(a, b).stop - panel_index.slice_indexer(a, b).start


span = pd.Series({t: _span_days(first[t], last[t]) for t in prices.columns})

coverage = pd.DataFrame({
    'start': first,
    'end': last,
    'n_obs': n_obs,
    'span_days': span,
})
coverage['coverage'] = (coverage['n_obs'] / coverage['span_days']).where(coverage['span_days'] > 0)
coverage['is_empty'] = coverage['n_obs'] == 0
coverage = coverage.sort_values('start')
coverage

In [ ]:
# Quick health summary.
empty = coverage[coverage['is_empty']]
nonempty = coverage[~coverage['is_empty']]

print(f'total ETFs:        {len(coverage)}')
print(f'  with data:       {len(nonempty)}')
print(f'  fully empty:     {len(empty)}')
print()
print(f'earliest start:    {nonempty["start"].min().date()}')
print(f'latest end:        {nonempty["end"].max().date()}')
print(f'latest start:      {nonempty["start"].max().date()}  (newest-listed ETF)')
print(f'earliest end:      {nonempty["end"].min().date()}  (earliest-delisted ETF)')
print()
print(f'median coverage:   {nonempty["coverage"].median():.3f}')
print(f'ETFs w/ gaps (<0.99 coverage): {(nonempty["coverage"] < 0.99).sum()}')
if len(empty):
    print()
    print('empty tickers:', list(empty.index))

In [ ]:
# Save the full table for reference.
out = ROOT / 'data' / 'processed' / 'etf_coverage.csv'
coverage.to_csv(out)
print('wrote', out)